In [5]:
-- Preview the first 10 rows of the survey dataset
-- Useful for understanding column structure and sample values

SELECT * FROM 'trafficData.csv'
-- Limit output to 10 records only
LIMIT 5;

,basket_items,geoLocationCity,ip_address,items_viewed,network,pages_visited,town,creation_date,Creator
0,None,Lusaka,165.56.185.101,None,ZAMTEL,https://morey.shop/category?category=Laptops%2...,None,"Nov 28, 2024 9:16 pm",(deleted thing)
1,"ASUS Lightweight 15.5"" Full HD Laptop, Acer As...",Berea,76.78.177.81,Acer Aspire V5 12-Inch Touchscreen,APOG-BEREACOLLEGE,https://morey.shop/product/acer-aspire-v5-12-i...,Mufulira District,"Nov 28, 2024 9:35 pm",beverlymusompo@gmail.com
2,JBL PARTYBOX ON THE GO,Lusaka,102.146.254.13,None,ZAIN-ZAMBIA,https://morey.shop/,None,"Nov 28, 2024 9:46 pm",kkunda99@gmail.com
3,JBL PARTYBOX ON THE GO,Lusaka,102.146.254.13,None,ZAIN-ZAMBIA,https://morey.shop/product?price=&color=&capac...,None,"Nov 28, 2024 9:49 pm",kkunda99@gmail.com
4,JBL PARTYBOX ON THE GO,Lusaka,102.146.254.13,None,ZAIN-ZAMBIA,https://morey.shop/category?search=JBL%20Party...,None,"Nov 28, 2024 9:50 pm",kkunda99@gmail.com


In [55]:
-- Convert the original creation_date (text) into a standard YYYY-MM-DD format
SELECT 
    STRFTIME(
        STRPTIME(creation_date, '%b %d, %Y %I:%M %p'),  -- parse text to timestamp
        '%Y-%m-%d'                                      -- format as date only
    ) AS creation_date,

    COUNT(*) AS count   -- count number of rows (records) per day

FROM trafficData.csv

-- Group results by the formatted date
GROUP BY 1   -- same as GROUP BY creation_date

-- Order results by count (highest first)
ORDER BY 2 DESC;   -- same as ORDER BY count DESC

,creation_date,count
0,2024-11-29,153
1,2024-12-01,102
2,2024-11-30,86
3,2024-12-02,84
4,2024-12-04,74
5,2024-12-03,27
6,2024-11-28,19
7,2024-12-05,13


In [56]:
-- First CTE: calculate authenticated and unauthenticated users
WITH temp1 AS (
    SELECT 
        -- Count distinct IP addresses where user doesn't have an email (unauthenticated users)
        COUNT(DISTINCT CASE WHEN Creator NOT LIKE '%@%' THEN ip_address END) AS unauthenticated_users,
        
        -- Count distinct users where user has an email (authenticated users)
        COUNT(DISTINCT CASE WHEN Creator LIKE '%@%' THEN Creator END) AS authenticated_users
    FROM trafficData.csv
),

-- Second CTE: find returning users (IP addresses that appear more than once)
returning_users_cte AS (
    SELECT COUNT(*) AS returning_users
    FROM (
        -- Select IPs that appear more than once in the dataset
        SELECT ip_address
        FROM trafficData.csv
        GROUP BY ip_address
        HAVING COUNT(*) > 1
    ) 
),

-- Third CTE: combine results from previous CTEs and compute total users
temp2 AS (
    SELECT 
        unauthenticated_users,
        authenticated_users,
        -- total users = authenticated + unauthenticated
        (unauthenticated_users + authenticated_users) AS total_users,
        returning_users
    FROM temp1, returning_users_cte
)

-- calculate percentages and display results
SELECT 
    authenticated_users,
    unauthenticated_users,
    total_users,
    returning_users,
    
    -- Percentage of returning users
    ROUND((returning_users * 100.0 / total_users), 2) AS returning_users_percent,
    
    -- Percentage of unauthenticated users
    ROUND((unauthenticated_users * 100.0 / total_users), 2) AS unauth_users_percent,
    
    -- Percentage of authenticated users
    ROUND((authenticated_users * 100.0 / total_users), 2) AS auth_users_percent
FROM temp2;

,authenticated_users,unauthenticated_users,total_users,returning_users,returning_users_percent,unauth_users_percent,auth_users_percent
0,14,145,159,83,52.2,91.19,8.81


In [57]:
/* Step 1: Create a daily summary of unique users */
WITH daily_visits AS (
    SELECT 
        -- Convert raw timestamp text into clean date format (YYYY-MM-DD)
        STRFTIME(
            STRPTIME(creation_date, '%b %d, %Y %I:%M %p'),
            '%Y-%m-%d'
        ) AS creation_date,

        -- Count unique users (IP addresses) per day
        COUNT(DISTINCT ip_address) AS visits

    FROM trafficData.csv

    -- Group by each day
    GROUP BY 1
)

/* Compute the average number of unique users per day */
SELECT 
    AVG(visits) AS avg_daily_visits  -- average of daily unique user counts
FROM daily_visits;

,avg_daily_visits
0,22.375


In [58]:
/*  Split basket_items into individual items per user */
WITH basket_items AS (
    SELECT 
        ip_address,

        -- Split comma-separated basket items into rows
        TRIM(UNNEST(STRING_TO_ARRAY(basket_items, ','))) AS item

    FROM trafficData.csv
),

/* Clean items and standardize text */
clean_items AS (
    SELECT 
        ip_address,
        item,

        -- Convert to lowercase for consistent matching
        LOWER(item) AS item_lower

    FROM basket_items
)

/* Classify each item into product categories */
SELECT DISTINCT 

    CASE 

        /* Mobile-related products */
        WHEN item_lower LIKE '%phone%' 
          OR item_lower LIKE '%samsung%' 
          OR item_lower LIKE '%mobile%' 
          OR item_lower LIKE '%cover%'
          OR item_lower LIKE '%charging%'
          OR item_lower LIKE '%nokia%'
          OR item_lower LIKE '%infinix%'
        THEN 'Mobile'

        /* Laptop-related products */
        WHEN item_lower LIKE '%intel%' 
          OR item_lower LIKE '%lenovo%' 
          OR item_lower LIKE '%macbook%' 
          OR item_lower LIKE '%laptop%' 
          OR item_lower LIKE '%dell%' 
          OR item_lower LIKE '%acer%' 
          OR item_lower LIKE '%asus%' 
          OR item_lower LIKE '%hp%' 
          OR item_lower LIKE '%neo%' 
        THEN 'Laptop'

        /* Smart watches */
        WHEN item_lower LIKE '%watch%' 
          OR item_lower LIKE '%smart%' 
          OR item_lower LIKE '%green-lion-lunar%' 
        THEN 'Smart Watch'

        /* Cameras */
        WHEN item_lower LIKE '%canon%' 
          OR item_lower LIKE '%camera%' 
        THEN 'Camera'

        /* Gaming products */
        WHEN item_lower LIKE '%playstation%' 
          OR item_lower LIKE '%xbox%' 
          OR item_lower LIKE '%controller%' 
          OR item_lower LIKE '%gaming%' 
        THEN 'Gaming'

        /* Audio products */
        WHEN item_lower LIKE '%jbl%' 
          OR item_lower LIKE '%airpod%'  
          OR item_lower LIKE '%speaker%' 
        THEN 'Sound'

        /* Power accessories */
        WHEN item_lower LIKE '%power%' 
          OR item_lower LIKE '%bank%'  
        THEN 'Power'

        /* Tablets */
        WHEN item_lower LIKE '%ipad%' 
          OR item_lower LIKE '%tablet%'  
        THEN 'Tablet'

        /* Everything else */
        ELSE 'Other'
    END AS category,

    -- Count number of items in each category
    COUNT(*) AS count

FROM clean_items

-- Remove empty items
WHERE item <> ''

-- Group results by category
GROUP BY category

-- Show most popular categories first
ORDER BY count DESC;

,category,count
0,Mobile,103
1,Laptop,55
2,Smart Watch,36
3,Other,33
4,Sound,13
5,Power,5
6,Gaming,1


In [59]:
/* Clean and normalize the dataset */
WITH avg_table AS (
    SELECT DISTINCT 

        -- user
        ip_address,

        -- Change raw text date into standard YYYY-MM-DD format
        STRFTIME(
            STRPTIME(creation_date, '%b %d, %Y %I:%M %p'),
            '%Y-%m-%d'
        ) AS creation_date,

        -- Split basket items into individual rows and remove extra spaces
        TRIM(UNNEST(STRING_TO_ARRAY(basket_items, ','))) AS basket_items

    FROM trafficData.csv

    -- Remove empty or null basket entries
    WHERE basket_items IS NOT NULL 
      AND basket_items <> ''
),

/* Count unique basket items per user per day */
user_day_counts AS (
    SELECT 
        ip_address,
        creation_date,

        -- Number of distinct basket items per user per day
        COUNT(DISTINCT basket_items) AS cnt

    FROM avg_table

    -- Keep only meaningful item strings
    WHERE LENGTH(basket_items) > 5

    GROUP BY ip_address, creation_date
),

/* Compute daily average basket size per user */
daily_avg AS (
    SELECT 
        creation_date,

        -- Average number of items per user for each day
        AVG(cnt) AS avg_items_per_day

    FROM user_day_counts

    GROUP BY creation_date
)

/* Final result - overall average across all days */
SELECT 
    ROUND(AVG(avg_items_per_day), 2) AS overall_avg_items_per_day
FROM daily_avg;

,overall_avg_items_per_day
0,2.29


In [60]:
/* Clean and standardize the dataset */
WITH cleaned AS (
    SELECT 
        ip_address,

        -- Creator (user identifier, may have an email or not)
        Creator,

        -- Basket items as raw text
        basket_items,

        -- Convert creation_date into standard YYYY-MM-DD format
        STRFTIME(
            STRPTIME(creation_date, '%b %d, %Y %I:%M %p'),
            '%Y-%m-%d'
        ) AS creation_date
    FROM trafficData.csv
),

/* Self-join to compare user activity across different days */
cleaned_2 AS (
    SELECT 
        a.ip_address,
        a.Creator AS creator,
        a.basket_items AS basket_items,

        -- First date (current row)
        a.creation_date AS creation_date_1,

        -- Second date (matched row from same user on different day)
        b.creation_date AS creation_date_2,

        -- Count how many matching pairs exist
        COUNT(*) AS cnt

    FROM cleaned a

    -- Join same user across different dates
    LEFT JOIN cleaned b
        ON a.ip_address = b.ip_address
        AND a.creation_date != b.creation_date

    -- Keep only valid basket entries
    WHERE 
        a.basket_items IS NOT NULL
        AND b.basket_items IS NOT NULL

    -- Group by user, creator, basket item, and both dates
    GROUP BY 
        a.ip_address,
        a.Creator,
        a.basket_items,
        a.creation_date,
        b.creation_date

    -- Keep only users with repeated activity
    HAVING COUNT(*) > 1

    -- Sort by highest repetition count
    ORDER BY cnt DESC
)

/* Calculate email vs non-email user proportions */
SELECT 
    -- Percentage of users with email in creator field
    ROUND(
        COUNT(DISTINCT CASE 
            WHEN creator LIKE '%@%' AND basket_items IS NOT NULL THEN creator 
        END) 
        / COUNT(DISTINCT creator) * 100, 2
    ) AS percent_with_email,

    -- Percentage of users without email in creator field
    ROUND(
        COUNT(DISTINCT CASE 
            WHEN creator NOT LIKE '%@%' AND basket_items IS NOT NULL THEN creator 
        END) 
        / COUNT(DISTINCT creator) * 100, 2
    ) AS percent_without_email

FROM cleaned_2;

,percent_with_email,percent_without_email
0,66.67,33.33


In [61]:
/* Find top cities by number of unique users */
SELECT 
    -- City where the user is located
    geoLocationCity,

    -- Count of unique users (IP addresses) per city
    COUNT(DISTINCT ip_address) AS count

FROM trafficData.csv

/* Filter out empty or null basket data */
WHERE basket_items IS NOT NULL 
  AND basket_items <> ''   -- ensure basket is not empty

/* Group results by city */
GROUP BY 1

/* Step 4: Sort cities by highest number of users */
ORDER BY 2 DESC

/* Step 5: Show only top 10 cities */
LIMIT 10;

,geoLocationCity,count
0,Lusaka,28
1,Lexington,2
2,Berea,2
3,Palermo,1
4,Ndola,1
5,Chisamba,1
6,Solwezi,1
7,Milan,1
8,Paris,1


In [62]:
/* Clean and standardize the dataset */
WITH cleaned AS (
    SELECT 
        ip_address,

        -- Basket items 
        basket_items,

        -- Change creation_date into standard YYYY-MM-DD format
        STRFTIME(
            STRPTIME(creation_date, '%b %d, %Y %I:%M %p'),
            '%Y-%m-%d'
        ) AS creation_date
    FROM trafficData.csv
)

/* Compare user activity across different days using self-join */
SELECT 
    -- User identifier
    a.ip_address,

    -- Basket items for that user
    a.basket_items,

    -- First date (current row)
    a.creation_date AS creation_date_1,

    -- Second date (same user, different day)
    b.creation_date AS creation_date_2,

    -- Count how many times this user appears on different days
    COUNT(*) AS cnt

FROM cleaned a

-- Self join to compare same user across different dates
LEFT JOIN cleaned b
    ON a.ip_address = b.ip_address
    AND a.creation_date != b.creation_date

/* Remove invalid rows */
WHERE 
    a.basket_items IS NOT NULL
    AND b.basket_items IS NOT NULL

/* Group by user, basket, and both dates */
GROUP BY 1, 2 , 3 , 4

/* Keep only users who have repeated visits */
HAVING COUNT(*) > 1

/* Show strongest repeated relationships first */
ORDER BY 5 DESC;

,ip_address,basket_items,creation_date_1,creation_date_2,cnt
0,76.78.177.74,"ASUS Lightweight 15.5"" Full HD Laptop, Acer As...",2024-11-29,2024-12-01,55
1,76.78.177.74,"ASUS Lightweight 15.5"" Full HD Laptop, Acer As...",2024-12-01,2024-11-29,44
2,197.212.210.204,Samsung S21,2024-11-29,2024-11-30,30
3,197.212.210.204,Samsung S21,2024-11-30,2024-11-29,20
4,76.78.177.74,iPhone 12,2024-12-01,2024-11-29,11
5,197.212.210.204,"Samsung S22 128GB , Samsung S20 Ultra 128GB",2024-11-30,2024-11-29,10
